# 07 — Summary & Takeaways

**LeadSense: Insurance Lead Acquisition Profitability**

This notebook consolidates findings from notebooks 01–06 into actionable insights. No new analysis — just the headline numbers and what to do with them.

## At a Glance — What This Project Found

If you've just dropped into this notebook and want the whole story in 30 seconds, here it is:

1. **The business is profitable, but only barely.** At £55 per lead, the average lead returns just enough premium to cover its cost — meaning a small dip in quality could flip the whole portfolio into the red.
2. **The averages hide huge variation.** Verified leads, desktop traffic, and brand keywords are clearly profitable. Unverified leads, smartphone traffic on broad keywords, and several specific keywords with zero sales are losing money every week.
3. **Statistical tests confirm the patterns are real.** The verified-vs-unverified gap and the keyword-group RPL differences are both significant — they're not noise.
4. **The ML model is weak (~0.58 AUC) — and that's still useful.** No model we tried beat 0.58 because the features just don't carry enough signal. But ranking leads by score lets us reach roughly 63% of conversions in the top half of the list, cutting effective CAC from ~£1,128 to ~£850.
5. **The clearest immediate action is to pause the zero-sale keywords.** Those are pure waste — no model needed.

The rest of this notebook walks through the supporting numbers.

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
leads = pd.read_parquet(ROOT / "data" / "leads_clean.parquet")

COST = 55
total = len(leads)
sold  = int(leads["converted"].sum())
rate  = sold / total
rev   = leads["premium"].sum()
rpl   = rev / total
net   = rpl - COST

print(f"Leads: {total:,}  |  Sold: {sold}  |  Conv rate: {rate:.1%}")
print(f"RPL: £{rpl:.2f}  |  Cost/Lead: £{COST}  |  Net/Lead: £{net:+.2f}")
print(f"Total Revenue: £{rev:,.0f}  |  Total Cost: £{total * COST:,.0f}  |  ROI: {(rev - total*COST)/(total*COST)*100:.1f}%")

Leads: 7,485  |  Sold: 367  |  Conv rate: 4.9%
RPL: £76.61  |  Cost/Lead: £55  |  Net/Lead: £+21.61
Total Revenue: £573,400  |  Total Cost: £411,675  |  ROI: 39.3%


## 1. The Business Is Profitable — But Barely

At £55 per lead, the portfolio is slightly profitable overall. But the average masks huge variation by segment.

## 2. Winners & Losers — Dimensional Insights

From NB03 and NB04, the clearest profit drivers:

| Dimension | Best Segment | Worst Segment | Insight |
|-----------|-------------|---------------|---------|
| **Verification** | Verified (higher RPL) | Unverified (lower conv) | Verified leads are worth paying more for |
| **Device** | Desktop | Smartphone | Desktop users convert at higher value |
| **Keyword Group** | Brand keywords | Generic broad terms | Brand traffic has clear intent → higher premiums |
| **Current Insurance** | Has insurance | No insurance | Switchers already understand the product |
| **Smoker** | Non-smoker | Smoker | Non-smokers get lower premiums but convert more readily |

### Zero-Sale Keywords
Several keywords with 15+ leads produced **zero** conversions — these should be paused immediately to stop wasting spend.

In [2]:
# Top 5 keyword groups by net/lead
kw = leads.groupby("keyword_group").agg(
    leads=("lead_id", "count"), sales=("converted", "sum"), revenue=("premium", "sum")
).reset_index()
kw["rpl"] = (kw["revenue"] / kw["leads"]).round(2)
kw["net"] = (kw["rpl"] - COST).round(2)
kw = kw.sort_values("net", ascending=False)
print("Keyword Group Profitability")
print(kw[["keyword_group", "leads", "sales", "rpl", "net"]].to_string(index=False))

Keyword Group Profitability
            keyword_group  leads  sales    rpl    net
             Brand: Other     89     11 252.59 197.59
     Price / Quote Intent    419     16  85.87  30.87
Generic: Health Insurance   1219     64  75.34  20.34
  Generic: Private Health   3484    182  75.11  20.11
              Brand: Bupa   1594     76  74.99  19.99
    Comparison / Research    488     14  74.85  19.85
                    Other    192      4  27.83 -27.17


## 3. ML Model — Honest Assessment

| Model | ROC-AUC | Notes |
|-------|---------|-------|
| Logistic Regression | ~0.58 | Baseline |
| LightGBM (default) | ~0.58 | No improvement |
| LightGBM (Optuna-tuned) | ~0.58 | Still ~0.58 |

**Why all models perform similarly (~0.58 AUC):**
- The available features (age, gender, device, keyword, etc.) carry only weak signal about conversion
- This isn't a model failure — it's a **feature limitation**. The data we have simply doesn't contain strong predictors
- To meaningfully improve, we'd need behavioural features: time-on-page, pages viewed, quote interactions, call duration

**But the model is still useful:**
- At threshold 0.12–0.18, the model achieves **best CAC of ~£850** vs ~£1,128 with no model
- The top 50% of scored leads captures ~63% of actual conversions
- Even marginal lift saves real money at scale

## 4. Deployment Stack

The project includes a production-ready serving layer:

| Component | File | Purpose |
|-----------|------|---------|
| Training script | `src/models/train.py` | Rebuilds model from staging data → saves to `app/artefacts/` |
| Scoring API | `app/main.py` | FastAPI with `/health` and `/score` endpoints |
| Dashboard | `app/dashboard.py` | Streamlit — KPIs, funnel, dimensional RPL, live lead scorer |
| Simulator | `app/simulate.py` | Generates realistic synthetic leads, optionally scores via API |
| Docker | `Dockerfile` | Wraps API + dashboard for single-container deployment |
| Tests | `tests/` | 13 tests covering API endpoints and model artefact integrity |

## 5. Recommendations

### Immediate Actions (Analytics-driven)
1. **Pause zero-sale keywords** — stop paying £55/lead for keywords that never convert
2. **Bid up on Desktop + Verified + Brand** segments — these are consistently above breakeven
3. **Bid down or exclude Smartphone + Unverified + Generic Broad** — these drag RPL below £55

### Model-Driven Actions
4. **Use the scoring model for lead prioritisation** — rank incoming leads, focus sales effort on top-scored leads first
5. **Set threshold at 0.12–0.18** for best CAC — accept lower recall for much lower cost-per-acquisition

### Next Steps (if this were a real engagement)
6. **Enrich features** — add behavioural data (time on site, pages, quote requests) to improve model beyond 0.58 AUC
7. **A/B test** — run model-scored vs. random allocation for 4 weeks to measure real lift
8. **Automate** — connect the scoring API to the CRM to auto-prioritise leads in real time

---

## Project Notebook Index

| # | Notebook | What it does |
|---|----------|-------------|
| 00 | `00_dataset_prep` | Anonymise raw data |
| 01 | `01_data_fetch` | Load CSV + generate synthetic spend |
| 02 | `02_clean_and_validate` | Raw → staging → mart (RPL, CAC, ROAS) |
| 03 | `03_eda_leads` | Lead profitability analysis by dimension |
| 04 | `04_hypothesis_testing` | Z-tests, Kruskal-Wallis, A/B sizing |
| 05 | `05_modeling` | LogReg → LightGBM → Optuna + MLflow |
| 06 | `06_threshold_and_business` | Threshold sweep, cumulative gains, CAC sim |
| 07 | `07_summary` | This notebook — findings & recommendations |

---

## What a Viewer Should Take Away

If you watched the whole video series and only remember three things, make it these:

- **Frame the project around a business question, not a metric.** "Are we profitable at £55 per lead?" is a better starting point than "can I get AUC > 0.8?". Every notebook in this project answered to that one question.
- **Honest results beat polished results.** The model topped out at 0.58 AUC and we showed it. Then we showed how to make it useful anyway. That arc — disappointment → reframing → win — is more instructive than another tutorial where everything works on the first try.
- **The decision layer is where ML becomes valuable.** Picking the right threshold, the right operating point, the right segments to act on — that's where the model meets the business. Skip that step and you've just built a number generator.

**What's coming next in the series:** moving the notebook logic into clean Python pipelines, adding data validation with Pandera, tracking experiments with MLflow, packaging with Docker, deploying to the cloud, and adding monitoring. The notebooks were the *thinking phase*. The next phase is making it ship.